# Lightweight Self-Supervised Text Representation Learning
## Google Colab / Kaggle Execution Launcher

**Project:** A Lightweight Self-Supervised Framework for Text Representation Learning Using Masked Latent Prediction.

This notebook verifies the runtime environment, runs the test suite, generates inspectable batches meeting `docs/INTERFACES.md`, executes representation geometry diagnostics, and tests durable checkpoint recovery across runtime resets.

> **Security Notice:** Never embed GitHub tokens or credentials in this notebook or commit them to git. When cloning a private repository, use Colab Secrets (`userdata.get('GH_TOKEN')`) or enter credentials interactively.

### 1. Hardware and Environment Inspection

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### 2. Repository Checkout and Dependency Installation
Mount Google Drive (optional for durable artifact persistence) and install tested requirements.

In [ ]:
# Optional: Mount Google Drive for durable checkpoint storage
# from google.colab import drive
# drive.mount('/content/drive')

!pip install --upgrade pip
!pip install -r requirements.txt

### 3. Verify Test Suite
Execute deterministic CPU unit tests covering the data pipeline, masking policies, pooling invariants, and geometry metrics.

In [ ]:
!python -m pytest tests/ -v

### 4. Inspect Batch Contract
Generate and inspect a tiny real batch strictly conforming to `docs/INTERFACES.md`.

In [ ]:
from src.data import get_bert_tokenizer, process_single_text, collate_batch, decode_masked_example, count_tokens_in_batch

tokenizer = get_bert_tokenizer("bert-base-uncased")
sample_text = "Undergraduate research on self-supervised text representation compares masked latent prediction with MLM."

item = process_single_text(sample_text, tokenizer, max_length=128, mask_ratio=0.15, mask_mode="independent", seed=42)
batch = collate_batch([item])
counts = count_tokens_in_batch(batch)

print("Contract Tensors:")
for k, v in batch.items():
    print(f"  {k:18s} shape={list(v.shape)} dtype={v.dtype}")

print(f"\nTokens counted: {counts}")
decoded = decode_masked_example(item["input_ids"], item["masked_input_ids"], item["prediction_mask"], tokenizer)
print(f"\nCorrupted Text:\n{decoded['corrupted_text']}")
print(f"Target Tokens: {decoded['target_tokens']}")

### 5. Corpus Statistics and Plots
Generate and display real corpus token length distribution and top token frequencies from WikiText.

In [ ]:
!python scripts/generate_artifacts.py

from IPython.display import Image, display
display(Image("results/token_length_distribution.png"))
display(Image("results/top_tokens_frequency.png"))

### 6. Representation Geometry Diagnostics
Evaluate RankMe (singular value effective rank), covariance eigenvalue rank, dimension variance, and pairwise cosine distributions.

In [ ]:
import json
with open("results/geometry_diagnostic_demo.json", "r") as f:
    geom_data = json.load(f)

print("Geometry Diagnostics Across Structural Regimes:")
for regime_name, metrics in geom_data["regimes"].items():
    print(f"\n--- Regime: {regime_name} ---")
    print(f"Status: {metrics['status']} (Degenerate: {metrics['is_degenerate']})")
    print(f"Effective Rank (Singular / RankMe): {metrics.get('effective_rank_singular', 'N/A')}")
    print(f"Effective Rank (Covariance):        {metrics.get('effective_rank_covariance', 'N/A')}")
    print(f"Mean Dimension Variance:          {metrics.get('mean_dimension_variance', 'N/A')}")
    print(f"Dead Dimension Fraction:          {metrics.get('dead_dimension_fraction', 'N/A')}")

### 7. Durable Checkpoint Saving and Reload Recovery Test
Verifies that a saved student encoder checkpoint can be reliably reloaded and frozen for evaluation without state corruption.

In [ ]:
import os
import torch
import torch.nn as nn
from src.evaluate import load_student_checkpoint, freeze_model_for_evaluation

# Minimal architecture matching project specification (hidden_dim=256, vocab=30522)
class StudentEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(30522, 256)
        self.proj = nn.Linear(256, 256)
    def forward(self, input_ids, attention_mask=None):
        return self.proj(self.embed(input_ids))

model1 = StudentEncoder()
checkpoint_payload = {
    "student_encoder": model1.state_dict(),
    "step": 500,
    "config": {"hidden_dim": 256, "max_length": 128, "vocab_size": 30522},
}

ckpt_dir = "results"
ckpt_path = os.path.join(ckpt_dir, "recovery_test_checkpoint.pt")
torch.save(checkpoint_payload, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

# Simulate fresh runtime recovery
model2 = StudentEncoder()
recovered_model = load_student_checkpoint(ckpt_path, model2)

# Verify parameters bitwise match
matched = True
for p1, p2 in zip(model1.parameters(), recovered_model.parameters()):
    if not torch.equal(p1, p2):
        matched = False
    if p2.requires_grad:
        print("ERROR: Recovered model parameter requires grad!")

print(f"Checkpoint recovery test: {'PASSED (weights match, model frozen)' if matched else 'FAILED'}")